In [ ]:
# ============================================================================
#  Resample ExtraSensory raw sensor data to a uniform 32 Hz grid
# ----------------------------------------------------------------------------
#  Input : raw_acc/<uuid>/<epoch>.m_raw_acc.dat     (60 users)
#          proc_gyro/<uuid>/<epoch>.m_proc_gyro.dat (57 users)
#          whitespace-separated, no header: timestamp[s]  x  y  z
#          every file = one recorded minute-window, sampled at a rate that
#          varies from file to file (acc ~33-50 Hz, gyro ~22-100 Hz).
#  Output: acc_32Hz/<uuid>/<epoch>.csv
#          gyro_32Hz/<uuid>/<epoch>.csv
#          header row "timestamp,x,y,z", exactly 32 Hz, one file per input file.
# ============================================================================
import os
import numpy as np
import pandas as pd
from concurrent.futures import ProcessPoolExecutor, as_completed
from scipy.signal import butter, filtfilt
from tqdm.auto import tqdm

# ------------------------------- configuration ------------------------------
TARGET_HZ = 32.0                 # desired output sampling rate
DT        = 1.0 / TARGET_HZ      # 0.03125 s between consecutive output samples

SOURCES = [
    {"in_dir": "raw_acc",   "out_dir": "acc_32Hz"},
    {"in_dir": "proc_gyro", "out_dir": "gyro_32Hz"},
]

# Length of each output window:
#   None      -> keep each file's own recorded span (no extrapolation; a 20 s
#                window becomes 641 samples, a 23 s window 737, etc.)
#   e.g. 20.0 -> force every file to exactly 20 s = 640 samples (edge values
#                are held constant if the file is shorter than that).
FIXED_WINDOW_SEC = None

# Files recorded ABOVE 32 Hz are being downsampled, so frequencies above the
# new 16 Hz Nyquist would alias. True = zero-phase Butterworth low-pass first.
# Set to False to use plain linear interpolation for every file.
ANTIALIAS   = True
AA_ORDER    = 4
AA_CUTOFF   = 0.9 * (TARGET_HZ / 2.0)   # 14.4 Hz

# Guards against corrupt timestamps (some files carry an all-zero padding row,
# and a few users log absolute epoch seconds instead of device uptime, so a
# single stray value can imply a span of decades). A window is split at any gap
# this large and only the longest continuous run is kept; smaller gaps are
# bridged by the interpolation.
GAP_FACTOR  = 200        # gap > 200x the median sample interval ...
MIN_GAP_SEC = 5.0        # ... and > 5 s  =>  treat as a break, not a dropout
MAX_OUT_SEC = 300.0      # refuse to emit a window longer than this

FLOAT_FMT   = "%.6f"    # microsecond / micro-g resolution, keeps files small
N_WORKERS   = max(1, min(24, (os.cpu_count() or 4)))
SKIP_EXISTING = True    # makes the job resumable after an interruption


# ------------------------------- core routine -------------------------------
def resample_file(in_path, out_path):
    """Read one raw .dat window, write it back out as a 32 Hz .csv.

    Returns (status, original_rate_hz, n_in, n_out).
    """
    try:
        raw = pd.read_csv(in_path, sep=r"\s+", header=None,
                          engine="c", dtype=np.float64).to_numpy()
    except Exception:
        return ("unreadable", np.nan, 0, 0)

    if raw.ndim != 2 or raw.shape[0] < 2 or raw.shape[1] < 4:
        return ("too_short", np.nan, len(raw), 0)

    t, xyz = raw[:, 0], raw[:, 1:4]

    # clean: drop NaN/inf rows, sort by time, drop duplicate timestamps
    good = np.isfinite(t) & np.isfinite(xyz).all(axis=1) & (t > 0)
    t, xyz = t[good], xyz[good]
    order = np.argsort(t, kind="stable")
    t, xyz = t[order], xyz[order]
    keep = np.concatenate(([True], np.diff(t) > 0))
    t, xyz = t[keep], xyz[keep]
    if len(t) < 2:
        return ("too_short", np.nan, len(t), 0)

    # keep the longest continuous run, so one bad timestamp (a zero-filled
    # padding row, a clock reset) cannot stretch the window to decades
    status = "ok"
    steps = np.diff(t)
    gap_limit = max(GAP_FACTOR * np.median(steps), MIN_GAP_SEC)
    breaks = np.flatnonzero(steps > gap_limit)
    if breaks.size:
        starts = np.concatenate(([0], breaks + 1))
        ends   = np.concatenate((breaks + 1, [len(t)]))
        longest = int(np.argmax(ends - starts))
        lo, hi = starts[longest], ends[longest]
        if hi - lo < len(t):
            status = "ok_trimmed"
        t, xyz = t[lo:hi], xyz[lo:hi]
        if len(t) < 2:
            return ("too_short", np.nan, len(t), 0)

    duration = t[-1] - t[0]
    fs_orig  = (len(t) - 1) / duration          # actual rate of this file

    # uniform 32 Hz time grid, starting at the file's first timestamp
    if FIXED_WINDOW_SEC is not None:
        n_out = int(round(FIXED_WINDOW_SEC * TARGET_HZ))
    else:
        n_out = int(np.floor(duration / DT)) + 1
    if n_out > int(MAX_OUT_SEC * TARGET_HZ):
        return ("bad_timestamps", fs_orig, len(t), 0)
    t_new = t[0] + np.arange(n_out) * DT

    src_t, src_xyz = t, xyz

    # downsampling -> low-pass first so nothing above 16 Hz folds back in
    if ANTIALIAS and fs_orig > TARGET_HZ * 1.01:
        n_u  = int(np.floor(duration * fs_orig)) + 1
        t_u  = t[0] + np.arange(n_u) / fs_orig  # uniform grid at the ORIGINAL rate
        u    = np.column_stack([np.interp(t_u, t, xyz[:, k]) for k in range(3)])
        wn   = min(AA_CUTOFF / (fs_orig / 2.0), 0.99)
        b, a = butter(AA_ORDER, wn)
        if len(t_u) > 3 * max(len(a), len(b)):  # filtfilt needs enough padding
            u = filtfilt(b, a, u, axis=0)
        src_t, src_xyz = t_u, u

    # linear interpolation onto the 32 Hz grid (this is the up-sampling step
    # for every file recorded below 32 Hz, and the re-gridding step otherwise)
    out = np.column_stack([t_new] +
                          [np.interp(t_new, src_t, src_xyz[:, k]) for k in range(3)])

    pd.DataFrame(out, columns=["timestamp", "x", "y", "z"]).to_csv(
        out_path, index=False, float_format=FLOAT_FMT)
    return (status, fs_orig, len(t), n_out)


def resample_user(args):
    """Resample every window of one user. Runs in a worker process."""
    in_dir, out_dir, uuid = args
    src = os.path.join(in_dir, uuid)
    dst = os.path.join(out_dir, uuid)
    os.makedirs(dst, exist_ok=True)

    counts, rates = {}, []
    for name in sorted(os.listdir(src)):
        in_path = os.path.join(src, name)
        if not os.path.isfile(in_path):
            continue
        # "1444079161.m_raw_acc.dat" -> "1444079161.csv"
        out_path = os.path.join(dst, name.split(".")[0] + ".csv")
        if SKIP_EXISTING and os.path.exists(out_path):
            counts["already_done"] = counts.get("already_done", 0) + 1
            continue
        status, fs, _, _ = resample_file(in_path, out_path)
        counts[status] = counts.get(status, 0) + 1
        if status.startswith("ok"):
            rates.append(fs)
    return uuid, counts, rates

In [ ]:
# ------------------------- run the conversion (both sensors) ----------------
import multiprocessing as mp
from collections import Counter

for cfg in SOURCES:
    in_dir, out_dir = cfg["in_dir"], cfg["out_dir"]
    users = sorted(u for u in os.listdir(in_dir)
                   if os.path.isdir(os.path.join(in_dir, u)))
    os.makedirs(out_dir, exist_ok=True)
    print(f"\n=== {in_dir} -> {out_dir} : {len(users)} users, {N_WORKERS} workers ===")

    totals, all_rates = Counter(), []
    jobs = [(in_dir, out_dir, u) for u in users]
    ctx = mp.get_context("fork")
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as pool:
        futures = [pool.submit(resample_user, j) for j in jobs]
        for fut in tqdm(as_completed(futures), total=len(futures), unit="user"):
            uuid, counts, rates = fut.result()
            totals.update(counts)
            all_rates.extend(rates)

    print("  files:", dict(totals))
    if all_rates:
        r = np.asarray(all_rates)
        print("  original rate (Hz): min %.1f  median %.1f  max %.1f  "
              "| %.1f%% of files were below %.0f Hz (up-sampled by interpolation)"
              % (r.min(), np.median(r), r.max(),
                 100.0 * np.mean(r < TARGET_HZ), TARGET_HZ))

# --------------------------------- verify -----------------------------------
print("\n=== verification ===")
for cfg in SOURCES:
    out_dir = cfg["out_dir"]
    users = sorted(u for u in os.listdir(out_dir)
                   if os.path.isdir(os.path.join(out_dir, u)))
    n_files = sum(len(os.listdir(os.path.join(out_dir, u))) for u in users)
    n_in = sum(len(os.listdir(os.path.join(cfg["in_dir"], u))) for u in users)
    print(f"{out_dir}: {len(users)} user folders, {n_files} csv files "
          f"(input had {n_in})")

    sample = os.path.join(out_dir, users[0],
                          sorted(os.listdir(os.path.join(out_dir, users[0])))[0])
    df = pd.read_csv(sample)
    step = np.diff(df["timestamp"].to_numpy())
    print(f"  sample {sample}: {len(df)} rows, columns {list(df.columns)}")
    print(f"  timestamp step = {step.mean():.8f} s  -> {1/step.mean():.4f} Hz "
          f"(max deviation {np.abs(step - DT).max():.2e} s)")
    display(df.head(3))

In [ ]:
# ============================================================================
#  Normalise accelerometer units:  put every user on the g scale
# ----------------------------------------------------------------------------
#  ExtraSensory did not use one unit convention across devices - some users'
#  accelerometer is logged in g (a phone at rest reads |a| ~ 1.0) and others in
#  m/s^2 (a phone at rest reads |a| ~ 9.81). Mixing the two makes every
#  scale-sensitive feature (mean, std, energy, FFT magnitude) ~9.8x larger for
#  one group, which a model happily uses to identify the *user* instead of the
#  activity. The gyroscope is rad/s everywhere and needs no correction.
#
#  A user is only rescaled when its unit is UNAMBIGUOUS: at least
#  CONSISTENCY_MIN of its files must sit in the gravity band for one scale or
#  the other. A user whose own files disagree is left untouched and reported,
#  because no single factor can fix it (see the manifest's "flagged" entry).
#
#  Rewrites acc_32Hz in place. Each file is written to a .tmp and then moved
#  over the original with os.replace(), which is atomic on Linux, so an
#  interruption can never leave a half-written CSV. A manifest records which
#  users are done, making the cell resumable and safe to re-run.
# ============================================================================
import json
import random
import multiprocessing as mp

ACC_DIR   = "acc_32Hz"
MANIFEST  = os.path.join(ACC_DIR, "_unit_normalisation.json")
G         = 9.80665          # m/s^2 per g
PROBE_N   = 60               # files per user used to identify the unit
BAND      = (0.5, 2.0)       # a resting/normal window's |a| in g
CONSISTENCY_MIN = 0.80       # fraction of files that must agree on the scale


def measure_user(u):
    """Return (user, median |a|, fraction of files reading g, fraction reading m/s^2)."""
    folder = os.path.join(ACC_DIR, u)
    files = [f for f in os.listdir(folder) if f.endswith(".csv")]
    rng = random.Random(u)                       # deterministic per user
    mags = []
    for name in rng.sample(files, min(PROBE_N, len(files))):
        df = pd.read_csv(os.path.join(folder, name))
        if len(df):
            mags.append(np.median(np.linalg.norm(df[["x", "y", "z"]].to_numpy(), axis=1)))
    if not mags:
        return u, np.nan, 0.0, 0.0
    m = np.asarray(mags)
    in_g   = float(np.mean((m > BAND[0])     & (m < BAND[1])))
    in_ms2 = float(np.mean((m > BAND[0] * G) & (m < BAND[1] * G)))
    return u, float(np.median(m)), in_g, in_ms2


def scale_user(args):
    """Divide one user's x/y/z by `factor`, rewriting each file atomically."""
    u, factor = args
    folder = os.path.join(ACC_DIR, u)
    n = 0
    for name in sorted(os.listdir(folder)):
        if not name.endswith(".csv"):
            continue
        path = os.path.join(folder, name)
        df = pd.read_csv(path)
        df[["x", "y", "z"]] = df[["x", "y", "z"]].to_numpy() / factor
        tmp = path + ".tmp"
        df.to_csv(tmp, index=False, float_format=FLOAT_FMT)
        os.replace(tmp, path)                    # atomic: old or new, never partial
        n += 1
    return u, n


users = sorted(u for u in os.listdir(ACC_DIR)
               if os.path.isdir(os.path.join(ACC_DIR, u)))
done = set(json.load(open(MANIFEST))["converted"]) if os.path.exists(MANIFEST) else set()

# ---- 1. identify each user's unit ------------------------------------------
ctx = mp.get_context("fork")
with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as pool:
    stats = {u: (med, g_, m_) for u, med, g_, m_ in
             tqdm(pool.map(measure_user, users), total=len(users),
                  desc="measuring", unit="user")}

todo    = [u for u, (_, _, m_) in stats.items() if m_ >= CONSISTENCY_MIN and u not in done]
already = [u for u, (_, g_, _) in stats.items() if g_ >= CONSISTENCY_MIN]
flagged = [u for u, (_, g_, m_) in stats.items()
           if g_ < CONSISTENCY_MIN and m_ < CONSISTENCY_MIN]

print(f"\n{len(already)} users already on the g scale, {len(todo)} to convert from m/s^2, "
      f"{len(flagged)} flagged as inconsistent")
for u in todo:
    print(f"  convert  {u}  |a| = {stats[u][0]:6.3f}  ->  {stats[u][0]/G:.3f}")
for u in flagged:
    med, g_, m_ = stats[u]
    print(f"  FLAGGED  {u}  |a| = {med:6.3f}  only {max(g_, m_)*100:.0f}% of its files "
          f"agree on a scale - left untouched")

# ---- 2. rewrite the unambiguous ones in place ------------------------------
if todo:
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as pool:
        futures = [pool.submit(scale_user, (u, G)) for u in todo]
        for fut in tqdm(as_completed(futures), total=len(futures),
                        desc="rescaling", unit="user"):
            u, n = fut.result()
            done.add(u)
            json.dump({"factor": G, "converted": sorted(done),
                       "flagged": {u_: "inconsistent accelerometer scale in the source "
                                       "data - exclude or handle separately"
                                   for u_ in flagged}},
                      open(MANIFEST, "w"), indent=1)   # checkpoint after each user
else:
    json.dump({"factor": G, "converted": sorted(done),
               "flagged": {u_: "inconsistent accelerometer scale in the source data - "
                               "exclude or handle separately" for u_ in flagged}},
              open(MANIFEST, "w"), indent=1)

# ---- 3. verify ---------------------------------------------------------------
with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as pool:
    after = {u: (med, g_) for u, med, g_, _ in pool.map(measure_user, users)}
ok = [u for u, (_, g_) in after.items() if g_ >= CONSISTENCY_MIN]
print(f"\n=== after ===\n  {len(ok)} of {len(users)} users are internally consistent on the g scale")
bad = {u: round(after[u][0], 3) for u in after if u not in ok}
print("  not on the g scale:", bad if bad else "none")

In [5]:
# ============================================================================
#  Count the files actually present in the resampled folders
# ============================================================================
import os

FOLDERS = ["acc_32Hz", "gyro_32Hz"]
SHOW_PER_USER = True          # set False for totals only

for folder in FOLDERS:
    if not os.path.isdir(folder):
        print(f"{folder}: MISSING")
        continue

    users = sorted(u for u in os.listdir(folder)
                   if os.path.isdir(os.path.join(folder, u)))
    counts = {u: sum(1 for f in os.listdir(os.path.join(folder, u))
                     if f.endswith(".csv")) for u in users}
    total = sum(counts.values())

    # anything in the folder that is not a per-user directory (e.g. the manifest)
    extras = [f for f in os.listdir(folder)
              if not os.path.isdir(os.path.join(folder, f))]

    print(f"\n{'=' * 62}\n{folder}\n{'=' * 62}")
    if SHOW_PER_USER:
        for i, u in enumerate(users, 1):
            print(f"  {i:3d}. {u}  {counts[u]:>6d}")
    print(f"  {'-' * 58}")
    print(f"  users            : {len(users)}")
    print(f"  csv files        : {total}")
    if counts:
        print(f"  files per user   : min {min(counts.values())}  "
              f"median {int(sorted(counts.values())[len(counts) // 2])}  "
              f"max {max(counts.values())}")
    if extras:
        print(f"  non-user entries : {extras}")

# cross-check against the sources these were built from
print(f"\n{'=' * 62}\ncross-check vs source folders\n{'=' * 62}")
for src, out in [("raw_acc", "acc_32Hz"), ("proc_gyro", "gyro_32Hz")]:
    if not (os.path.isdir(src) and os.path.isdir(out)):
        continue
    out_users = sorted(u for u in os.listdir(out)
                       if os.path.isdir(os.path.join(out, u)))
    n_out = sum(sum(1 for f in os.listdir(os.path.join(out, u)) if f.endswith(".csv"))
                for u in out_users)
    # count only the source users that still exist in the output
    n_src = sum(len(os.listdir(os.path.join(src, u))) for u in out_users)
    dropped = sorted(set(os.listdir(src)) - set(out_users))
    status = "match" if n_out == n_src else "MISMATCH"
    print(f"  {src:10s} -> {out:10s}  {n_src} -> {n_out}  [{status}]"
          + (f"   dropped users: {len(dropped)}" if dropped else ""))


acc_32Hz
    1. 00EABED2-271D-49D8-B599-1D4A09240601    2287
    2. 098A72A5-E3E5-4F54-A152-BBDA0DF7B694    6808
    3. 0A986513-7828-4D53-AA1F-E02D6DF9561B    3960
    4. 0BFC35E2-4817-4865-BFA7-764742302A2D    3090
    5. 0E6184E1-90C0-48EE-B25A-F1ECB7B9714E    7513
    6. 1155FF54-63D3-4AB2-9863-8385D0BD0A13    2685
    7. 11B5EC4D-4133-4289-B475-4E737182A406    8845
    8. 136562B6-95B2-483D-88DC-065F28409FD2    6218
    9. 1538C99F-BA1E-4EFB-A949-6C7C47701B20    6549
   10. 1DBB0F6F-1F81-4A50-9DF4-CD62ACFA4842    7371
   11. 24E40C4C-A349-4F9F-93AB-01D00FB994AF    4771
   12. 27E04243-B138-4F40-A164-F40B60165CF3    4925
   13. 2C32C23E-E30C-498A-8DD2-0EFB9150A02E    8516
   14. 33A85C34-CFE4-4732-9E73-0A7AC861B27A    6164
   15. 3600D531-0C55-44A7-AE95-A7A38519464E    5203
   16. 40E170A7-607B-4578-AF04-F021C3B0384A    7648
   17. 481F4DD2-7689-43B9-A2AA-C8772227162B    6690
   18. 4E98F91F-4654-42EF-B908-A3389443F2E7    3244
   19. 4FC32141-E888-4BFF-8804-12559A491D8C    4979
  

In [ ]:
# ============================================================================
#  Merge accelerometer + gyroscope onto the accelerometer's timestamps
# ----------------------------------------------------------------------------
#  One parquet file per user (56 users - the gyroscope set), containing every
#  minute-window of that user concatenated in chronological order:
#
#     window   int64    epoch id of the minute-window = the source filename,
#                       and the join key to feature_labels/<uuid>...csv.gz
#     timestamp float64 the ACCELEROMETER clock, unchanged
#     acc_x/y/z float64 accelerometer, as-is (already 32 Hz, g units)
#     gyro_x/y/z float64 gyroscope resampled onto the accelerometer timestamps
#     gyro_extrapolated bool  True where the gyro value was extrapolated rather
#                       than interpolated. Set ADD_EXTRAP_FLAG=False to drop it.
#
#  Gyro is linearly INTERPOLATED at accelerometer timestamps inside its recorded
#  range. Outside that range it is EXTRAPOLATED with an autoregressive model:
#  an AR(p) is fitted to the nearest AR_TRAIN_N gyro samples and iterated
#  forward on the gyro's own 32 Hz grid until the grid covers the accelerometer
#  timestamps, after which the normal interpolation applies.
#
#  The AR coefficients come from Yule-Walker solved by Levinson-Durbin, which
#  guarantees a stable model (all roots inside the unit circle). That matters
#  here: a stable AR forecast decays towards the signal's own mean as it runs
#  on, so a long gap degrades to "no rotation" rather than diverging. A linear
#  fit has no such bound - on this data it reached 203 rad/s, ~10x anything the
#  sensors ever measured.
# ============================================================================
import json
import multiprocessing as mp
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

ACC_DIR   = "acc_32Hz"
GYRO_DIR  = "gyro_32Hz"
MERGE_DIR = "merged_acc_gyro"

AR_ORDER         = 16      # AR model order
AR_TRAIN_N       = 256     # gyro samples (8 s at 32 Hz) the model is fitted on
CHUNK_WINDOWS    = 400     # windows per parquet row-group (bounds memory)
MERGE_WORKERS    = 12      # fewer than N_WORKERS: each holds a chunk in memory
ADD_EXTRAP_FLAG  = True
COMPRESSION      = "zstd"

_COLS = ["window", "timestamp", "acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
SCHEMA = pa.schema([(_COLS[0], pa.int64())] + [(c, pa.float64()) for c in _COLS[1:]]
                   + ([("gyro_extrapolated", pa.bool_())] if ADD_EXTRAP_FLAG else []))


def ar_fit_yw(x, p):
    """AR(p) coefficients by Yule-Walker / Levinson-Durbin. Returns (coeffs, mean).

    Yule-Walker is used rather than least squares because it always yields a
    stable model, which is what keeps the forecast bounded.
    """
    x = np.asarray(x, float)
    mu = x.mean()
    xc = x - mu
    n = len(xc)
    p = min(p, n - 1)
    if p < 1:
        return np.zeros(0), mu
    r = np.correlate(xc, xc, mode="full")[n - 1:n + p] / n     # autocorrelation
    if r[0] <= 1e-30:                                          # constant signal
        return np.zeros(0), mu
    a = np.zeros(p + 1)
    E = r[0]
    for k in range(1, p + 1):
        acc = r[k] - (np.dot(a[1:k], r[k - 1:0:-1]) if k > 1 else 0.0)
        kap = acc / E
        new = a.copy()
        new[k] = kap
        if k > 1:
            new[1:k] = a[1:k] - kap * a[k - 1:0:-1]
        a = new
        E *= (1 - kap ** 2)
        if E <= 1e-30:
            break
    return a[1:], mu


def ar_forecast(x, p, n_ahead):
    """Iterate an AR(p) fitted on x forward n_ahead steps."""
    if n_ahead <= 0:
        return np.zeros(0)
    a, mu = ar_fit_yw(x, p)
    if len(a) == 0:
        return np.full(n_ahead, mu)                            # flat at the mean
    hist = list(np.asarray(x, float)[-len(a):] - mu)
    out = np.empty(n_ahead)
    for i in range(n_ahead):
        nxt = float(np.dot(a, hist[::-1]))
        out[i] = nxt
        hist.append(nxt)
        hist.pop(0)
    return out + mu


def gyro_on_acc_grid(tg, vg3, ta):
    """Gyro (3 axes) at accelerometer timestamps.

    Inside the gyro's range: linear interpolation. Outside: the gyro series is
    first extended on its own uniform grid by AR forecasting (backwards at the
    start, forwards at the end), then interpolated as normal.
    """
    dt = float(np.median(np.diff(tg))) if len(tg) > 1 else 1.0 / TARGET_HZ
    before = ta < tg[0]
    after = ta > tg[-1]
    n_b = int(np.ceil((tg[0] - ta[0]) / dt)) if before.any() else 0
    n_a = int(np.ceil((ta[-1] - tg[-1]) / dt)) if after.any() else 0

    t_ext = tg
    if n_b:
        t_ext = np.concatenate([tg[0] - dt * np.arange(n_b, 0, -1), t_ext])
    if n_a:
        t_ext = np.concatenate([t_ext, tg[-1] + dt * np.arange(1, n_a + 1)])

    out = {}
    for ax, v in vg3.items():
        head = ar_forecast(v[:AR_TRAIN_N][::-1], AR_ORDER, n_b)[::-1] if n_b else np.zeros(0)
        tail = ar_forecast(v[-AR_TRAIN_N:], AR_ORDER, n_a) if n_a else np.zeros(0)
        v_ext = np.concatenate([head, v, tail])
        out[f"gyro_{ax}"] = np.interp(ta, t_ext, v_ext)
    return out, (before | after)


def merge_user(u):
    """Merge one user's windows into a single parquet file. Runs in a worker."""
    acc_files  = {f for f in os.listdir(os.path.join(ACC_DIR, u))  if f.endswith(".csv")}
    gyro_files = {f for f in os.listdir(os.path.join(GYRO_DIR, u)) if f.endswith(".csv")}
    shared = sorted(acc_files & gyro_files)       # chronological: filenames are epochs

    os.makedirs(MERGE_DIR, exist_ok=True)
    out_path = os.path.join(MERGE_DIR, f"{u}.parquet")
    tmp_path = out_path + ".tmp"

    st = {"user": u, "windows": 0, "skipped_no_gyro": len(acc_files - gyro_files),
          "rows": 0, "rows_extrap": 0, "max_abs_real": 0.0, "max_abs_extrap": 0.0}
    buf, writer = [], None

    def flush():
        nonlocal buf, writer
        if not buf:
            return
        df = pd.concat(buf, ignore_index=True)
        if writer is None:
            writer = pq.ParquetWriter(tmp_path, SCHEMA, compression=COMPRESSION)
        writer.write_table(pa.Table.from_pandas(df, schema=SCHEMA, preserve_index=False))
        buf = []

    for name in shared:
        a = pd.read_csv(os.path.join(ACC_DIR, u, name))
        g = pd.read_csv(os.path.join(GYRO_DIR, u, name))
        if len(a) == 0 or len(g) < 2:
            continue
        ta = a["timestamp"].to_numpy()
        tg = g["timestamp"].to_numpy()

        cols, mask = gyro_on_acc_grid(tg, {ax: g[ax].to_numpy() for ax in "xyz"}, ta)

        d = pd.DataFrame({
            "window": np.int64(name.split(".")[0]),
            "timestamp": ta,
            "acc_x": a["x"].to_numpy(), "acc_y": a["y"].to_numpy(), "acc_z": a["z"].to_numpy(),
            **cols,
        })
        if ADD_EXTRAP_FLAG:
            d["gyro_extrapolated"] = mask

        gm = np.linalg.norm(np.column_stack([cols["gyro_x"], cols["gyro_y"], cols["gyro_z"]]), axis=1)
        if (~mask).any():
            st["max_abs_real"] = max(st["max_abs_real"], float(gm[~mask].max()))
        if mask.any():
            st["max_abs_extrap"] = max(st["max_abs_extrap"], float(gm[mask].max()))

        st["windows"] += 1
        st["rows"] += len(d)
        st["rows_extrap"] += int(mask.sum())
        buf.append(d)
        if len(buf) >= CHUNK_WINDOWS:
            flush()

    flush()
    if writer is not None:
        writer.close()
        os.replace(tmp_path, out_path)            # atomic
    return st


users = sorted(set(u for u in os.listdir(GYRO_DIR) if os.path.isdir(os.path.join(GYRO_DIR, u)))
               & set(u for u in os.listdir(ACC_DIR) if os.path.isdir(os.path.join(ACC_DIR, u))))
os.makedirs(MERGE_DIR, exist_ok=True)
print(f"merging {len(users)} users -> {MERGE_DIR}/  "
      f"(AR({AR_ORDER}) on {AR_TRAIN_N} samples, {MERGE_WORKERS} workers)")

ctx = mp.get_context("fork")
with ProcessPoolExecutor(max_workers=MERGE_WORKERS, mp_context=ctx) as pool:
    stats = list(tqdm(pool.map(merge_user, users), total=len(users), unit="user"))

rows    = sum(s["rows"] for s in stats)
extrap  = sum(s["rows_extrap"] for s in stats)
wins    = sum(s["windows"] for s in stats)
skipped = sum(s["skipped_no_gyro"] for s in stats)
print(f"\n  files written    : {len(stats)}")
print(f"  windows merged   : {wins}   (skipped, no gyro counterpart: {skipped})")
print(f"  rows             : {rows:,}")
print(f"  extrapolated rows: {extrap:,}  ({100 * extrap / max(rows, 1):.1f}%)")

worst = sorted(stats, key=lambda s: -s["rows_extrap"] / max(s["rows"], 1))[:5]
print("\n  users with the most extrapolated gyro (share of rows | "
      "max |gyro| measured -> extrapolated):")
for s in worst:
    print(f"    {s['user'][:8]}  {100 * s['rows_extrap'] / max(s['rows'], 1):5.1f}%   "
          f"{s['max_abs_real']:10.3f} -> {s['max_abs_extrap']:12.3f}")

# ---- verify one file round-trips -------------------------------------------
sample = os.path.join(MERGE_DIR, f"{users[0]}.parquet")
df = pd.read_parquet(sample)
print(f"\n=== {sample} ===")
print(f"  {len(df):,} rows x {len(df.columns)} cols, {os.path.getsize(sample) / 1e6:.1f} MB")
print(f"  columns: {list(df.columns)}")
print(f"  windows: {df['window'].nunique()}   NaNs: {int(df.isna().sum().sum())}")
display(df.head(3))

In [ ]:
# ============================================================================
#  Attach activity labels to the merged data
# ----------------------------------------------------------------------------
#  Labels are per minute-window, keyed by the epoch `timestamp` in
#    feature_labels/<uuid>.features_labels.csv.gz   (5 of the 7 activities)
#    original_labels/<uuid>.original_labels.csv.gz  (the 2 standing activities)
#  which is exactly the `window` column of merged_acc_gyro.
#
#  Output: one parquet per user in labeled_acc_gyro/, 9 columns -
#    window, timestamp, acc_x, acc_y, acc_z, gyro_x, gyro_y, gyro_z,
#    Activity_Labels (int8, 1-7)
#
#  Verified on this data: no window carries more than one of the 7 labels, so
#  the single integer is unambiguous - no tie-breaking rule is applied or
#  needed. Windows carrying NONE of the 7 are dropped. NaN in a label column
#  means "not reported" and is treated as 0.
# ============================================================================
import multiprocessing as mp
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

MERGE_DIR  = "merged_acc_gyro"
FEAT_DIR   = "feature_labels"
ORIG_DIR   = "original_labels"
LABEL_DIR  = "labeled_acc_gyro"

# code -> (source column, human name). Order is the ranking 1..7.
LABELS = [
    (1, "label:LYING_DOWN",                   "Lying down"),
    (2, "label:SITTING",                      "Sitting"),
    (3, "label:FIX_walking",                  "Walking"),
    (4, "label:FIX_running",                  "Running"),
    (5, "label:BICYCLING",                    "Bicycling"),
    (6, "original_label:STANDING_IN_PLACE",   "Standing in place"),
    (7, "original_label:STANDING_AND_MOVING", "Standing and moving"),
]
FEAT_COLS = [c for _, c, _ in LABELS if c.startswith("label:")]
ORIG_COLS = [c for _, c, _ in LABELS if c.startswith("original_label:")]
CODE_NAME = {k: n for k, _, n in LABELS}

DATA_COLS  = ["window", "timestamp", "acc_x", "acc_y", "acc_z",
              "gyro_x", "gyro_y", "gyro_z"]
OUT_SCHEMA = pa.schema([("window", pa.int64())]
                       + [(c, pa.float64()) for c in DATA_COLS[1:]]
                       + [("Activity_Labels", pa.int8())])
BATCH_ROWS   = 500_000     # rows per read/write batch (bounds memory)
LABEL_WORKERS = 12


def window_codes(u):
    """Map {window epoch -> activity code 1..7} for one user.

    Windows with none of the 7 labels are simply absent from the map.
    """
    f = pd.read_csv(f"{FEAT_DIR}/{u}.features_labels.csv.gz",
                    usecols=["timestamp"] + FEAT_COLS)
    o = pd.read_csv(f"{ORIG_DIR}/{u}.original_labels.csv.gz",
                    usecols=["timestamp"] + ORIG_COLS)
    m = f.merge(o, on="timestamp", how="outer")

    cols = [c for _, c, _ in LABELS]
    A = np.nan_to_num(m[cols].to_numpy(dtype="float64"), nan=0.0) > 0.5
    n_active = A.sum(axis=1)
    if (n_active > 1).any():                       # never happens on this data;
        raise ValueError(f"{u}: {int((n_active > 1).sum())} multi-label windows")
    keep = n_active == 1
    codes = A[keep].argmax(axis=1) + 1             # column order == code order
    return dict(zip(m.loc[keep, "timestamp"].to_numpy(dtype="int64"),
                    codes.astype("int8")))


def label_user(u):
    """Write one user's labelled parquet. Runs in a worker process."""
    codes = window_codes(u)
    src = f"{MERGE_DIR}/{u}.parquet"
    out = f"{LABEL_DIR}/{u}.parquet"
    tmp = out + ".tmp"

    st = {"user": u, "rows_in": 0, "rows_out": 0,
          "windows_in": set(), "windows_out": set(),
          "per_class": {k: 0 for k, _, _ in LABELS},
          "per_class_win": {k: set() for k, _, _ in LABELS}}
    writer = None
    pf = pq.ParquetFile(src)
    for batch in pf.iter_batches(batch_size=BATCH_ROWS, columns=DATA_COLS):
        df = batch.to_pandas()
        st["rows_in"] += len(df)
        st["windows_in"].update(df["window"].unique().tolist())

        lab = df["window"].map(codes)              # NaN where unlabelled
        df = df[lab.notna()].copy()
        if df.empty:
            continue
        df["Activity_Labels"] = lab[lab.notna()].astype("int8").to_numpy()

        st["rows_out"] += len(df)
        st["windows_out"].update(df["window"].unique().tolist())
        for k, c in df["Activity_Labels"].value_counts().items():
            st["per_class"][int(k)] += int(c)
        for k, w in df.groupby("Activity_Labels")["window"].unique().items():
            st["per_class_win"][int(k)].update(w.tolist())

        if writer is None:
            writer = pq.ParquetWriter(tmp, OUT_SCHEMA, compression="zstd")
        writer.write_table(pa.Table.from_pandas(df, schema=OUT_SCHEMA,
                                                preserve_index=False))
    if writer is not None:
        writer.close()
        os.replace(tmp, out)                       # atomic
    st["windows_in"] = len(st["windows_in"])
    st["windows_out"] = len(st["windows_out"])
    st["per_class_win"] = {k: len(v) for k, v in st["per_class_win"].items()}
    st["classes_present"] = {k for k, v in st["per_class_win"].items() if v}
    return st


users = sorted(f[:-len(".parquet")] for f in os.listdir(MERGE_DIR)
               if f.endswith(".parquet"))
os.makedirs(LABEL_DIR, exist_ok=True)
print(f"labelling {len(users)} users -> {LABEL_DIR}/")

ctx = mp.get_context("fork")
with ProcessPoolExecutor(max_workers=LABEL_WORKERS, mp_context=ctx) as pool:
    stats = list(tqdm(pool.map(label_user, users), total=len(users), unit="user"))

rows_in  = sum(s["rows_in"] for s in stats)
rows_out = sum(s["rows_out"] for s in stats)
win_in   = sum(s["windows_in"] for s in stats)
win_out  = sum(s["windows_out"] for s in stats)
print(f"\n  users written : {sum(1 for s in stats if s['rows_out'] > 0)}"
      f"  (empty after labelling: {[s['user'][:8] for s in stats if s['rows_out'] == 0]})")
print(f"  windows       : {win_in:,} -> {win_out:,}  "
      f"({win_in - win_out:,} dropped, none of the 7 labels)")
print(f"  rows          : {rows_in:,} -> {rows_out:,}")

print("\n  class distribution:")
print(f"    {'code':>4}  {'activity':22s} {'windows':>10s} {'rows':>14s}  "
      f"{'share':>7s}  {'users':>7s}")
totals = {k: sum(s["per_class"][k] for s in stats) for k, _, _ in LABELS}
win_tot = {k: sum(s["per_class_win"][k] for s in stats) for k, _, _ in LABELS}
n_users = {k: sum(1 for s in stats if k in s["classes_present"]) for k, _, _ in LABELS}
for k, _, name in LABELS:
    print(f"    {k:>4}  {name:22s} {win_tot[k]:10,} {totals[k]:14,}  "
          f"{100 * totals[k] / max(rows_out, 1):6.2f}%  {n_users[k]:4d}/{len(stats)}")

# ---- verify ----------------------------------------------------------------
sample = f"{LABEL_DIR}/{users[0]}.parquet"
df = pd.read_parquet(sample)
print(f"\n=== {sample} ===")
print(f"  {len(df):,} rows x {len(df.columns)} cols, {os.path.getsize(sample) / 1e6:.1f} MB")
print(f"  columns: {list(df.columns)}")
print(f"  labels present: {sorted(df['Activity_Labels'].unique().tolist())}   NaNs: {int(df.isna().sum().sum())}")
display(df.head(3))

In [ ]:
# ============================================================================
#  Cut the labelled data into fixed-length 4 s segments
# ----------------------------------------------------------------------------
#  labeled_acc_gyro/ holds variable-length minute-windows (127-1536 rows).
#  Models need a fixed input size, so each window is cut into segments of
#  SEG_LEN samples. Segments are stored with 50% overlap (STRIDE = SEG_LEN/2),
#  which is the training default; evaluation needs NON-overlapping segments,
#  recovered by filtering  seg_start % SEG_LEN == 0  - no regeneration needed.
#
#  Output: segmented_4s/<uuid>.parquet, one row per sample, with
#     seg_id      int32   segment index within the user (0..n_seg-1)
#     seg_start   int16   offset of the segment inside its parent window,
#                         so any coarser stride can be filtered out later
#     window      int64   parent minute-window (label + CV join key)
#     timestamp   float64
#     acc_x/y/z, gyro_x/y/z  float64
#     Activity_Labels int8
#
#  Every segment has exactly SEG_LEN rows. The tail of a window that does not
#  fill a whole segment is DROPPED rather than padded - padding would recreate
#  the flat-line artefact that corrupts variance and energy features.
#  labeled_acc_gyro/ is left untouched: it is the only source that allows the
#  window size or stride to be changed later.
# ============================================================================
import multiprocessing as mp
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

SRC_DIR = "labeled_acc_gyro"
SEG_DIR = "segmented_4s"

SEG_LEN = 128              # 4.0 s at 32 Hz
STRIDE  = 64               # 50% overlap (training default)
SEG_WORKERS = 8            # each worker holds one user in memory

VAL_COLS = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
SEG_SCHEMA = pa.schema([("seg_id", pa.int32()), ("seg_start", pa.int16()),
                        ("window", pa.int64()), ("timestamp", pa.float64())]
                       + [(c, pa.float64()) for c in VAL_COLS]
                       + [("Activity_Labels", pa.int8())])


def segment_user(u):
    """Cut one user's windows into fixed-length segments."""
    d = pd.read_parquet(f"{SRC_DIR}/{u}.parquet")
    # rows are written in window order, so windows are contiguous blocks
    codes = pd.factorize(d["window"], sort=False)[0]
    starts = np.flatnonzero(np.r_[True, np.diff(codes) != 0])
    lens = np.diff(np.r_[starts, len(d)])

    # absolute row index at which each segment begins, and its offset in-window
    abs_start, in_window = [], []
    for ws, n in zip(starts, lens):
        if n < SEG_LEN:
            continue                                   # window too short: dropped
        off = np.arange(0, n - SEG_LEN + 1, STRIDE)
        abs_start.append(ws + off)
        in_window.append(off)
    st = {"user": u, "windows": len(starts), "windows_used": len(abs_start),
          "segments": 0, "per_class": {}}
    if not abs_start:
        return st

    abs_start = np.concatenate(abs_start)
    in_window = np.concatenate(in_window).astype("int16")
    n_seg = len(abs_start)

    # expand every segment into its SEG_LEN row positions in one shot
    rows = (abs_start[:, None] + np.arange(SEG_LEN)[None, :]).ravel()
    out = d.iloc[rows].reset_index(drop=True)
    out.insert(0, "seg_start", np.repeat(in_window, SEG_LEN))
    out.insert(0, "seg_id", np.repeat(np.arange(n_seg, dtype="int32"), SEG_LEN))

    tmp = f"{SEG_DIR}/{u}.parquet.tmp"
    pq.write_table(pa.Table.from_pandas(out, schema=SEG_SCHEMA, preserve_index=False),
                   tmp, compression="zstd")
    os.replace(tmp, f"{SEG_DIR}/{u}.parquet")           # atomic

    lab = out["Activity_Labels"].to_numpy()[::SEG_LEN]  # one label per segment
    st["segments"] = n_seg
    st["per_class"] = {int(k): int(v) for k, v in zip(*np.unique(lab, return_counts=True))}
    return st


users = sorted(f[:-len(".parquet")] for f in os.listdir(SRC_DIR) if f.endswith(".parquet"))
os.makedirs(SEG_DIR, exist_ok=True)
print(f"segmenting {len(users)} users -> {SEG_DIR}/  "
      f"(SEG_LEN={SEG_LEN} = {SEG_LEN / TARGET_HZ:.1f} s, STRIDE={STRIDE})")

ctx = mp.get_context("fork")
with ProcessPoolExecutor(max_workers=SEG_WORKERS, mp_context=ctx) as pool:
    stats = list(tqdm(pool.map(segment_user, users), total=len(users), unit="user"))

NAMES = {1: "Lying down", 2: "Sitting", 3: "Walking", 4: "Running",
         5: "Bicycling", 6: "Standing in place", 7: "Standing and moving"}
n_seg = sum(s["segments"] for s in stats)
w_all = sum(s["windows"] for s in stats)
w_use = sum(s["windows_used"] for s in stats)
print(f"\n  users written : {sum(1 for s in stats if s['segments'] > 0)}")
print(f"  windows       : {w_use:,} used, {w_all - w_use:,} too short for one segment")
print(f"  segments      : {n_seg:,}   ({n_seg * SEG_LEN:,} rows)")

print(f"\n  {'code':>4}  {'activity':22s} {'segments (50% ovl)':>19s} {'non-overlap':>13s}  {'share':>7s}")
for k in range(1, 8):
    c = sum(s["per_class"].get(k, 0) for s in stats)
    print(f"  {k:>4}  {NAMES[k]:22s} {c:19,} {c // 2:13,}  {100 * c / max(n_seg, 1):6.2f}%")

# ---- verify ----------------------------------------------------------------
sample = f"{SEG_DIR}/{users[0]}.parquet"
df = pd.read_parquet(sample)
sizes = df.groupby("seg_id").size()
print(f"\n=== {sample} ===")
print(f"  {len(df):,} rows, {df['seg_id'].nunique():,} segments, "
      f"{os.path.getsize(sample) / 1e6:.1f} MB")
print(f"  every segment is exactly {SEG_LEN} rows: {bool((sizes == SEG_LEN).all())}")
print(f"  one label per segment (no mixed segments): "
      f"{bool((df.groupby('seg_id')['Activity_Labels'].nunique() == 1).all())}")
print(f"  non-overlapping subset: {int((df['seg_start'] % SEG_LEN == 0).sum()) // SEG_LEN:,} segments")
display(df.head(3))

In [ ]:
# ============================================================================
#  Build train / validation / test splits -> updated_cv_5_folds/
# ----------------------------------------------------------------------------
#  cv_5_folds/ provides a subject-wise 5-fold split of the original 60 users,
#  stratified by device platform, with every user in exactly one test fold.
#  It gives only train+test; a validation set is needed for early stopping and
#  hyper-parameter choices, and 4 of the 60 users are absent from this pipeline
#  (3 have no gyroscope, 1 was dropped for inconsistent accelerometer units).
#
#  This cell therefore:
#    1. intersects every fold list with the users present in segmented_4s/
#    2. carves N_VAL validation users out of each fold's training pool,
#       REQUIRING at least MIN_RARE_USERS users of each rare class so that
#       early stopping on macro-F1 is not driven by an absent class
#    3. prefers modest rare-class contributors for validation, keeping the
#       heavy ones (e.g. 797D145F owns 26% of all Running) in training
#
#  Output: updated_cv_5_folds/
#     fold_<i>_{train,val,test}_uuids.txt   one UUID per line
#     splits.json                           same lists + provenance + coverage
#     summary.txt                           the printed coverage table
#
#  Deterministic: same SEED -> same split. Splits are lists of UUIDs; no data
#  is moved or copied - load segmented_4s/<uuid>.parquet for the users you want.
# ============================================================================
import json
import multiprocessing as mp
import numpy as np
import pandas as pd

SEG_DIR = "segmented_4s"
CV_SRC  = "cv_5_folds"
CV_OUT  = "updated_cv_5_folds"

N_VAL           = 8          # validation users per fold
RARE            = [4, 5]     # Running, Bicycling
MIN_RARE_USERS  = 2          # per rare class, in every validation set
MIN_RARE_SEGS   = 20         # "modest contributor" threshold
SEED            = 1000
SEG_LEN_ROWS    = 128        # rows per segment, to subsample labels cheaply

NAMES = {1: "Lying down", 2: "Sitting", 3: "Walking", 4: "Running",
         5: "Bicycling", 6: "Standing in place", 7: "Standing and moving"}


def user_class_counts(u):
    """Segments per class for one user."""
    d = pd.read_parquet(f"{SEG_DIR}/{u}.parquet", columns=["Activity_Labels"])
    lab = d["Activity_Labels"].to_numpy()[::SEG_LEN_ROWS]     # one per segment
    k, v = np.unique(lab, return_counts=True)
    return u, {int(a): int(b) for a, b in zip(k, v)}


def read_list(name):
    p = os.path.join(CV_SRC, name)
    return [l.strip() for l in open(p)] if os.path.exists(p) else []


available = sorted(f[:-len(".parquet")] for f in os.listdir(SEG_DIR)
                   if f.endswith(".parquet"))
ctx = mp.get_context("fork")
with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as pool:
    counts = dict(tqdm(pool.map(user_class_counts, available),
                       total=len(available), desc="counting", unit="user"))

android = set()
for i in range(5):
    android |= set(read_list(f"fold_{i}_test_android_uuids.txt"))
    android |= set(read_list(f"fold_{i}_train_android_uuids.txt"))

os.makedirs(CV_OUT, exist_ok=True)
splits, rows = {}, []
for i in range(5):
    test = sorted((set(read_list(f"fold_{i}_test_android_uuids.txt"))
                   | set(read_list(f"fold_{i}_test_iphone_uuids.txt"))) & set(available))
    train_pool = sorted((set(read_list(f"fold_{i}_train_android_uuids.txt"))
                         | set(read_list(f"fold_{i}_train_iphone_uuids.txt"))) & set(available))

    val = []
    for k in RARE:                              # guarantee rare-class coverage
        have = sorted((u for u in train_pool
                       if counts[u].get(k, 0) > 0 and u not in val),
                      key=lambda u: counts[u][k])
        modest = [u for u in have if counts[u][k] >= MIN_RARE_SEGS]
        val += [u for u in (modest if len(modest) >= MIN_RARE_USERS else have)[:MIN_RARE_USERS]
                if u not in val]
    rest = [u for u in train_pool if u not in val]
    rng = np.random.default_rng(SEED + i)
    if N_VAL > len(val):
        val += list(rng.choice(rest, size=N_VAL - len(val), replace=False))
    val = sorted(val)
    train = sorted(u for u in train_pool if u not in val)

    assert not (set(train) & set(val)), "train/val overlap"
    assert not (set(train) & set(test)), "train/test overlap"
    assert not (set(val) & set(test)), "val/test overlap"
    for k in RARE:
        assert sum(1 for u in val if counts[u].get(k, 0) > 0) >= MIN_RARE_USERS, \
            f"fold {i}: rare class {k} missing from validation"

    splits[i] = {"train": train, "val": val, "test": test}
    for part in ("train", "val", "test"):
        with open(os.path.join(CV_OUT, f"fold_{i}_{part}_uuids.txt"), "w") as fh:
            fh.write("\n".join(splits[i][part]) + "\n")
    rows.append({"fold": i,
                 **{f"n_{p}": len(splits[i][p]) for p in ("train", "val", "test")},
                 **{f"seg_{p}": sum(sum(counts[u].values()) for u in splits[i][p])
                    for p in ("train", "val", "test")},
                 **{f"{p}_run": sum(counts[u].get(4, 0) for u in splits[i][p])
                    for p in ("train", "val", "test")}})

json.dump({"seed": SEED, "n_val": N_VAL, "source": CV_SRC, "data": SEG_DIR,
           "min_rare_users": MIN_RARE_USERS, "rare_classes": RARE,
           "users_available": len(available),
           "users_missing_from_cv": sorted(
               set().union(*[set(read_list(f"fold_{i}_test_android_uuids.txt"))
                             | set(read_list(f"fold_{i}_test_iphone_uuids.txt"))
                             | set(read_list(f"fold_{i}_train_android_uuids.txt"))
                             | set(read_list(f"fold_{i}_train_iphone_uuids.txt"))
                             for i in range(5)]) - set(available)),
           "folds": {str(i): splits[i] for i in splits}},
          open(os.path.join(CV_OUT, "splits.json"), "w"), indent=1)

df = pd.DataFrame(rows)
tot = df[["seg_train", "seg_val", "seg_test"]].sum(axis=1)
for p in ("train", "val", "test"):
    df[f"%seg_{p}"] = (100 * df[f"seg_{p}"] / tot).round(1)
    df[f"%usr_{p}"] = (100 * df[f"n_{p}"] / df[["n_train", "n_val", "n_test"]].sum(axis=1)).round(1)

summary = (f"updated_cv_5_folds  (seed={SEED}, n_val={N_VAL}, "
           f"{len(available)} users from {SEG_DIR})\n\n"
           + df[["fold", "n_train", "n_val", "n_test",
                 "%usr_train", "%usr_val", "%usr_test",
                 "%seg_train", "%seg_val", "%seg_test",
                 "train_run", "val_run", "test_run"]].to_string(index=False))
open(os.path.join(CV_OUT, "summary.txt"), "w").write(summary + "\n")
print("\n" + summary)

# every user must be tested exactly once, across all folds
seen = [u for i in splits for u in splits[i]["test"]]
print(f"\n  users tested exactly once: {len(seen) == len(set(seen)) == len(available)}"
      f"  ({len(set(seen))}/{len(available)})")
print(f"  files written: {len(os.listdir(CV_OUT))} in {CV_OUT}/")

In [ ]:
# ============================================================================
#  Build balanced, augmented training sets per fold -> balanced_folds/
# ----------------------------------------------------------------------------
#  Training data only is rebalanced. Validation and test keep the NATURAL class
#  distribution and use non-overlapping segments, otherwise the metrics would
#  describe a distribution that does not exist.
#
#  UNDERSAMPLING (majority classes)
#    - stride filter: majority classes keep only non-overlapping segments
#      (seg_start % SEG_LEN == 0); the rest keep the stored 50% overlap
#    - equal per-user quota with water-filling redistribution, so a user who
#      contributes 26% of a class cannot dominate it. This attacks the
#      per-user concentration, not just the class ratio.
#
#  AUGMENTATION (minority classes), all label-preserving:
#    - rotation about the ESTIMATED GRAVITY AXIS. A free 3D rotation would move
#      gravity in the sensor frame and can turn Sitting into something that
#      looks like Lying down while keeping the Sitting label. Rotating about
#      gravity leaves the posture cue intact and varies only the heading.
#    - a small free rotation (<= SMALL_ROT_DEG) for orientation tolerance
#    - time warping: smooth non-linear time distortion = different cadence
#    - scaling of the DYNAMIC component only; scaling total acceleration would
#      make gravity read something other than 1 g
#    - jitter proportional to each channel's own std
#
#  Augmentation is capped at MAX_AUG_FACTOR x the real segments: inflating
#  ~6k Running segments to 50k would be 8 near-copies of each, which overfits.
#  Residual imbalance is left for class weights in the loss.
#
#  Output per fold in balanced_folds/fold_<i>/:
#    X_train (N,SEG_LEN,6) float32, y_train int8, u_train int16, aug_train bool
#    X_val / X_test and their y_, u_, plus w_test (window ids, for aggregating
#    segment predictions back to minute level)
#    norm_mean.npy / norm_std.npy  - per-channel stats fitted on REAL TRAINING
#    segments only, to be applied unchanged to val and test
# ============================================================================
import json
import multiprocessing as mp
import numpy as np
import pandas as pd

SEG_DIR  = "segmented_4s"
CV_DIR   = "updated_cv_5_folds"
OUT_DIR  = "balanced_folds"

SEG_LEN          = 128
CHANNELS         = ["acc_x", "acc_y", "acc_z", "gyro_x", "gyro_y", "gyro_z"]
TARGET_PER_CLASS = 50_000      # cap per class in the training set
MAX_AUG_FACTOR   = 6.0         # never synthesise more than 5x the real count
NON_OVERLAP_ONLY = [1, 2]      # majority classes: drop the 50% overlap
AUG_SEED         = 7

SMALL_ROT_DEG    = 15.0        # free-rotation perturbation
WARP_KNOTS       = 4           # control points for the time warp
WARP_SIGMA       = 0.2
SCALE_SIGMA      = 0.10
JITTER_FRAC      = 0.02        # x each channel's std

NAMES = {1: "Lying down", 2: "Sitting", 3: "Walking", 4: "Running",
         5: "Bicycling", 6: "Standing in place", 7: "Standing and moving"}


# ----------------------------------------------------------------- loading --
def load_user(u, non_overlap_classes=(), non_overlap_all=False):
    """Return (X (n,SEG_LEN,6) float32, y, seg_start, window) for one user."""
    d = pd.read_parquet(f"{SEG_DIR}/{u}.parquet",
                        columns=["seg_id", "seg_start", "window",
                                 "Activity_Labels"] + CHANNELS)
    n_seg = len(d) // SEG_LEN
    X = d[CHANNELS].to_numpy(dtype="float32").reshape(n_seg, SEG_LEN, 6)
    y = d["Activity_Labels"].to_numpy()[::SEG_LEN]
    st = d["seg_start"].to_numpy()[::SEG_LEN]
    w = d["window"].to_numpy()[::SEG_LEN]
    keep = np.ones(n_seg, bool)
    if non_overlap_all:
        keep &= (st % SEG_LEN == 0)
    else:
        for k in non_overlap_classes:                 # thin the majority only
            keep &= ~((y == k) & (st % SEG_LEN != 0))
    return X[keep], y[keep], st[keep], w[keep]


def user_labels(u):
    """Cheap pass: labels + seg_start only, for planning the quotas."""
    d = pd.read_parquet(f"{SEG_DIR}/{u}.parquet",
                        columns=["seg_start", "Activity_Labels"])
    return u, d["Activity_Labels"].to_numpy()[::SEG_LEN], d["seg_start"].to_numpy()[::SEG_LEN]


# ------------------------------------------------------------ augmentation --
def _rot_matrix(axis, angle):
    """Rodrigues rotation about a unit axis."""
    axis = axis / (np.linalg.norm(axis) + 1e-12)
    K = np.array([[0, -axis[2], axis[1]],
                  [axis[2], 0, -axis[0]],
                  [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * (K @ K)


def augment_segment(seg, rng):
    """One label-preserving augmentation of a (SEG_LEN, 6) segment."""
    acc, gyr = seg[:, :3].copy(), seg[:, 3:].copy()

    # 1. rotation about the estimated gravity axis (heading change)
    g = acc.mean(axis=0)
    if np.linalg.norm(g) > 1e-6:
        R = _rot_matrix(g, rng.uniform(-np.pi, np.pi))
        acc, gyr = acc @ R.T, gyr @ R.T

    # 2. small free rotation (orientation tolerance)
    ax = rng.normal(size=3)
    R2 = _rot_matrix(ax, np.deg2rad(rng.uniform(-SMALL_ROT_DEG, SMALL_ROT_DEG)))
    acc, gyr = acc @ R2.T, gyr @ R2.T

    # 3. time warp (cadence change)
    knots = np.linspace(0, SEG_LEN - 1, WARP_KNOTS + 2)
    offs = np.r_[0.0, rng.normal(0, WARP_SIGMA * SEG_LEN / WARP_KNOTS, WARP_KNOTS), 0.0]
    warped = np.sort(np.clip(knots + offs, 0, SEG_LEN - 1))
    src = np.interp(np.arange(SEG_LEN), knots, warped)
    idx = np.arange(SEG_LEN)
    acc = np.column_stack([np.interp(src, idx, acc[:, c]) for c in range(3)])
    gyr = np.column_stack([np.interp(src, idx, gyr[:, c]) for c in range(3)])

    # 4. scale the dynamic part only - gravity must stay at 1 g
    grav = acc.mean(axis=0)
    acc = grav + (acc - grav) * rng.normal(1.0, SCALE_SIGMA)
    gyr = gyr * rng.normal(1.0, SCALE_SIGMA)

    out = np.concatenate([acc, gyr], axis=1)
    sd = out.std(axis=0, keepdims=True)
    out = out + rng.normal(0.0, 1.0, out.shape) * sd * JITTER_FRAC   # 5. jitter
    return out.astype("float32")


# ------------------------------------------------------------- quota logic --
def water_fill(avail, target):
    """Equal per-user quota, redistributing what short users cannot supply."""
    avail = np.asarray(avail, dtype=np.int64)
    take = np.zeros_like(avail)
    remaining, active = int(min(target, avail.sum())), avail > 0
    while remaining > 0 and active.any():
        share = max(1, remaining // int(active.sum()))
        give = np.minimum(share, avail - take)
        give[~active] = 0
        if give.sum() == 0:
            break
        if give.sum() > remaining:                     # trim the last round
            order = np.flatnonzero(give)
            for j in order:
                if remaining <= 0:
                    give[j] = 0
                elif give[j] > remaining:
                    give[j] = remaining
                remaining -= give[j]
            take += give
            break
        take += give
        remaining -= int(give.sum())
        active = (avail - take) > 0
    return take


# ------------------------------------------------------------- build a fold --
def build_fold(i):
    rd = lambda p: [l.strip() for l in open(f"{CV_DIR}/fold_{i}_{p}_uuids.txt") if l.strip()]
    tr_u, va_u, te_u = rd("train"), rd("val"), rd("test")
    out = f"{OUT_DIR}/fold_{i}"
    os.makedirs(out, exist_ok=True)
    rng = np.random.default_rng(AUG_SEED + i)
    rep = {"fold": i, "n_train_users": len(tr_u)}

    # ---- plan: how many segments to take per (user, class) ----
    ctx = mp.get_context("fork")
    with ProcessPoolExecutor(max_workers=N_WORKERS, mp_context=ctx) as pool:
        lab = {u: (y, s) for u, y, s in pool.map(user_labels, tr_u)}
    avail = {}
    for k in NAMES:
        avail[k] = np.array([int(((y == k) & ((s % SEG_LEN == 0) if k in NON_OVERLAP_ONLY
                                              else np.ones(len(s), bool))).sum())
                             for u in tr_u for y, s in [lab[u]]])
    quota = {k: water_fill(avail[k], TARGET_PER_CLASS) for k in NAMES}

    # ---- collect the real training segments ----
    Xs, ys, us = [], [], []
    for j, u in enumerate(tr_u):
        Xu, yu, su, _ = load_user(u, non_overlap_classes=NON_OVERLAP_ONLY)
        for k in NAMES:
            q = int(quota[k][j])
            if q <= 0:
                continue
            idx = np.flatnonzero(yu == k)
            if len(idx) > q:                      # spread the pick over the session
                idx = idx[np.linspace(0, len(idx) - 1, q).round().astype(int)]
            Xs.append(Xu[idx]); ys.append(yu[idx]); us.append(np.full(len(idx), j, "int16"))
    X = np.concatenate(Xs); y = np.concatenate(ys); uu = np.concatenate(us)
    del Xs, ys, us
    rep["real_per_class"] = {int(k): int((y == k).sum()) for k in NAMES}

    # ---- augment the classes that fall short ----
    Xa, ya, ua = [], [], []
    for k in NAMES:
        have = int((y == k).sum())
        want = min(TARGET_PER_CLASS, int(have * MAX_AUG_FACTOR)) - have
        if have == 0 or want <= 0:
            continue
        pool_idx = np.flatnonzero(y == k)
        # round-robin over users so augmentation does not amplify one person
        order = np.argsort(uu[pool_idx], kind="stable")
        pool_idx = pool_idx[order]
        pick = pool_idx[np.arange(want) % len(pool_idx)]
        Xa.append(np.stack([augment_segment(X[p], rng) for p in pick]))
        ya.append(np.full(want, k, y.dtype)); ua.append(uu[pick])
    if Xa:
        X = np.concatenate([X] + Xa); y = np.concatenate([y] + ya)
        aug = np.concatenate([np.zeros(len(uu), bool)] + [np.ones(len(a), bool) for a in ya])
        uu = np.concatenate([uu] + ua)
    else:
        aug = np.zeros(len(y), bool)

    perm = rng.permutation(len(y))                 # shuffle once, at write time
    X, y, uu, aug = X[perm], y[perm], uu[perm], aug[perm]

    # normalisation statistics: REAL training segments only
    real = X[~aug].reshape(-1, 6)
    np.save(f"{out}/norm_mean.npy", real.mean(0).astype("float32"))
    np.save(f"{out}/norm_std.npy", (real.std(0) + 1e-8).astype("float32"))
    del real
    np.save(f"{out}/X_train.npy", X); np.save(f"{out}/y_train.npy", y.astype("int8"))
    np.save(f"{out}/u_train.npy", uu); np.save(f"{out}/aug_train.npy", aug)
    rep["final_per_class"] = {int(k): int((y == k).sum()) for k in NAMES}
    rep["aug_per_class"] = {int(k): int(((y == k) & aug).sum()) for k in NAMES}
    rep["n_train"] = int(len(y))
    del X, y, uu, aug

    # ---- validation and test: non-overlapping, natural distribution ----
    for part, users in (("val", va_u), ("test", te_u)):
        Xs, ys, us, ws = [], [], [], []
        for j, u in enumerate(users):
            Xu, yu, _, wu = load_user(u, non_overlap_all=True)
            Xs.append(Xu); ys.append(yu); us.append(np.full(len(yu), j, "int16")); ws.append(wu)
        Xp, yp = np.concatenate(Xs), np.concatenate(ys)
        np.save(f"{out}/X_{part}.npy", Xp)
        np.save(f"{out}/y_{part}.npy", yp.astype("int8"))
        np.save(f"{out}/u_{part}.npy", np.concatenate(us))
        np.save(f"{out}/w_{part}.npy", np.concatenate(ws))
        rep[f"n_{part}"] = int(len(yp))
        rep[f"{part}_per_class"] = {int(k): int((yp == k).sum()) for k in NAMES}
        del Xs, ys, us, ws, Xp, yp

    json.dump({"users": {"train": tr_u, "val": va_u, "test": te_u}, **rep},
              open(f"{out}/report.json", "w"), indent=1)
    return rep


os.makedirs(OUT_DIR, exist_ok=True)
reports = [build_fold(i) for i in tqdm(range(5), desc="folds", unit="fold")]

print(f"\n{'':22s}" + "".join(f"{'fold '+str(r['fold']):>14s}" for r in reports))
for k in NAMES:
    line = f"{NAMES[k]:22s}"
    for r in reports:
        line += f"{r['final_per_class'][k]:9,}{'+' + str(r['aug_per_class'][k]//1000) + 'k' if r['aug_per_class'][k] else '':>5s}"
    print(line)
print(f"{'TRAIN TOTAL':22s}" + "".join(f"{r['n_train']:14,}" for r in reports))
print(f"{'val (natural)':22s}" + "".join(f"{r['n_val']:14,}" for r in reports))
print(f"{'test (natural)':22s}" + "".join(f"{r['n_test']:14,}" for r in reports))
r0 = reports[0]
mx = max(r0["final_per_class"].values()); mn = min(r0["final_per_class"].values())
print(f"\n  fold 0 imbalance ratio after balancing: {mx / max(mn, 1):.1f}:1   (was 120:1)")
print(f"  augmented share of fold 0 training set: "
      f"{100 * sum(r0['aug_per_class'].values()) / r0['n_train']:.1f}%")

In [ ]:
# ============================================================================
#  Convert balanced_folds labels from 1-7 to 0-6
# ----------------------------------------------------------------------------
#  PyTorch CrossEntropyLoss, Keras sparse_categorical_crossentropy and XGBoost
#  all require targets in [0, n_classes-1]. With 7 output units and labels 1-7,
#  PyTorch raises "Target 7 is out of bounds"; giving the model 8 output units
#  instead makes it run but wastes unit 0 on a class that never occurs, and
#  argmax can still emit it. So the encoding is shifted here, once.
#
#  ONLY balanced_folds/*/y_*.npy are changed. The upstream parquet files
#  (labeled_acc_gyro/, segmented_4s/) keep the original 1-7 Activity_Labels,
#  which is the scheme the class ranking was defined in and the one you see
#  when inspecting the data. balanced_folds/label_map.json records both.
#
#  Idempotent: a fold whose labels are already 0-based is skipped, so re-running
#  this cell cannot shift the labels twice.
# ============================================================================
import json
import numpy as np

OUT_DIR = "balanced_folds"
NAMES_1BASED = {1: "Lying down", 2: "Sitting", 3: "Walking", 4: "Running",
                5: "Bicycling", 6: "Standing in place", 7: "Standing and moving"}

folds = sorted(d for d in os.listdir(OUT_DIR)
               if os.path.isdir(os.path.join(OUT_DIR, d)) and d.startswith("fold_"))
changed, skipped = [], []

for fold in folds:
    d = os.path.join(OUT_DIR, fold)
    for part in ("train", "val", "test"):
        p = os.path.join(d, f"y_{part}.npy")
        y = np.load(p)
        lo, hi = int(y.min()), int(y.max())
        if lo == 0:
            skipped.append(f"{fold}/{part}")
            continue
        if not (lo >= 1 and hi <= 7):
            raise ValueError(f"{p}: unexpected label range [{lo}, {hi}]")
        tmp = p + ".tmp.npy"
        np.save(tmp, (y - 1).astype("int8"))
        os.replace(tmp, p)                       # atomic
        changed.append(f"{fold}/{part}")

    # keep the fold's own report readable alongside the new encoding
    rp = os.path.join(d, "report.json")
    if os.path.exists(rp):
        r = json.load(open(rp))
        r["label_encoding"] = "y_*.npy are 0-6; the per_class dicts below are 1-7"
        json.dump(r, open(rp, "w"), indent=1)

json.dump({"encoding": "0-based",
           "classes": {str(k - 1): v for k, v in NAMES_1BASED.items()},
           "original_1based": {str(k): v for k, v in NAMES_1BASED.items()},
           "note": ("balanced_folds/*/y_*.npy use 0-6. Upstream parquet "
                    "(labeled_acc_gyro/, segmented_4s/) use 1-7. "
                    "model_class = Activity_Labels - 1")},
          open(os.path.join(OUT_DIR, "label_map.json"), "w"), indent=1)

print(f"converted: {len(changed)} arrays   already 0-based: {len(skipped)}")
print("\nverification:")
for fold in folds:
    d = os.path.join(OUT_DIR, fold)
    rng = []
    for part in ("train", "val", "test"):
        y = np.load(os.path.join(d, f"y_{part}.npy"))
        rng.append(f"{part} [{int(y.min())}-{int(y.max())}] n={len(y):,}")
    print(f"  {fold}: " + "  ".join(rng))

y = np.load(os.path.join(OUT_DIR, folds[0], "y_train.npy"))
c = np.bincount(y, minlength=7)
print(f"\n  fold_0 train counts by new index:")
for i in range(7):
    print(f"    {i} = {NAMES_1BASED[i + 1]:22s} {c[i]:8,}")
print(f"\n  dtype {y.dtype}, classes {sorted(set(y.tolist()))}")

In [16]:
import numpy as np
import pandas as pd

arr = np.load("balanced_folds/fold_0/u_test.npy")

print("Shape:", arr.shape)
print("Dimensions:", arr.ndim)
print("Data type:", arr.dtype)
df=pd.DataFrame(arr)
print(df.shape)
print(df.head(10))

Shape: (241576,)
Dimensions: 1
Data type: int16
(241576, 1)
   0
0  0
1  0
2  0
3  0
4  0
5  0
6  0
7  0
8  0
9  0
